# Soup Kernel Challenge — T4 benchmark (Decentrathon 6.0)

**Session options → Accelerator: `GPU T4 x1`** (mandatory).

This notebook runs the official harness on a Kaggle T4:
1. base `soup train` (Qwen2.5-1.5B-Instruct, 4-bit NF4, LoRA r=16, batch 4, seq 512, 64 steps, fp16),
2. the same run with `solution.apply()` — the fused Triton cross-entropy/entropy/accuracy kernels,
3. prints `runs/result.json` + `runs/checks.json` (speedup, peak memory, eval parity, kernel share).

Everything (harness, task, data, solution) comes from the submission repo, so the run is reproducible from this page alone.

In [ ]:
# 1) deps: torch/triton come with the Kaggle T4 image; pin the rest exactly as in the local env
!pip install -q 'transformers<4.46' 'peft==0.12.0' 'bitsandbytes==0.43.3' \
    'accelerate>=0.30' 'datasets>=2.20' 'huggingface_hub>=0.22' 'sentencepiece' 'protobuf' 'soup-cli==0.77.0'
!python -c "import torch, triton, transformers, peft, bitsandbytes; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0)); print('triton', triton.__version__)"
assert torch.cuda.is_available(), "GPU T4 is not visible — set Accelerator in Session options"

In [ ]:
# 2) fetch the submission (harness/ + task/ + data/ + solution/)
!git clone --depth 1 https://github.com/adilzhanw/soup-kernel-solution /kaggle/working/repo
!ls /kaggle/working/repo

In [ ]:
# 3) base run: plain soup train, no solution (timed from outside by the harness)
# ~64 steps of Qwen2.5-1.5B 4-bit on a T4: roughly 10-20 min
%cd /kaggle/working/repo
%env -u USE_OURS
!python harness/run_task.py --config task/task.yaml --only base --timeout 7200

In [ ]:
# 4) candidate run: same command, but solution.apply() is active (fused Triton CE)
%cd /kaggle/working/repo
%env USE_OURS=1
!python harness/run_task.py --config task/task.yaml --only solution --timeout 7200

In [ ]:
# 5) results: metrics, ratios (speedup / memory), and the 10-rule check table
!python - <<'PY'
import json, os
base = 'runs/base_1/metrics.json'
sol  = 'runs/solution_1/metrics.json'
for p in (base, sol):
    print('=' * 20, p, '=' * 20)
    m = json.load(open(p))
    keep = {k: m[k] for k in ('exit_code', 'steps', 'median_step_s', 'timed_loop_s',
                              'torch_max_allocated_mib', 'own_kernel_ms', 'triton_files') if k in m}
    print(json.dumps(keep, indent=2))
mb = json.load(open(base)); ms = json.load(open(sol))
if mb.get('timed_loop_s') and ms.get('timed_loop_s'):
    print('\nSPEEDUP (loop):  %.3fx' % (mb['timed_loop_s'] / ms['timed_loop_s']))
print('PEAK ALLOC MiB:    base %s  solution %s' % (mb.get('torch_max_allocated_mib'), ms.get('torch_max_allocated_mib')))
PY

In [ ]:
# 6) rules check (needs a combined result.json: run this cell only if you used the
#    single-command variant below; otherwise the jury runs it on the merged result)
# !python harness/check_rules.py

### Single-command variant (simpler, replaces cells 3–5)

```bash
cd /kaggle/working/repo && python harness/run_task.py --config task/task.yaml --timeout 7200 && python harness/check_rules.py
```

It runs base → solution back-to-back, writes `runs/result.json`, and `check_rules.py` prints the pass/fail table for all 10 checks (completion, log/metrics presence, `OWN` kernels, ≥5% Triton share, `torch.allclose` parity, ≥1.10× speedup, eval parity, ≥25% peak-memory reduction, banned-lib scan).